<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/master/homeworks/hw2/CNN_MNIST_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let's build a simple neural network to classify images from the FashionMNIST dataset.

**1. Import Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

*Checking for GPU Availability*

This code checks if a CUDA-enabled GPU is available and sets the `device` accordingly. If no GPU is available, it defaults to the CPU.

In [2]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


**2. Data Preparation**

In [3]:
# Define a transform to convert images to tensors
transform = transforms.ToTensor()

# Download and load the training data
train_set = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True)

# Download and load the test data
test_set = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.94MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 190kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.57MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.3MB/s]


**3. Neural Network Model**

In [34]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding = 1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding = 1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding = 1)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(in_features=28*28*128, out_features=128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.flat(x)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x

**4. Training loop with selection of parameters**

In [35]:
def train_and_evaluate(number_of_epochs, lr):

  model = SimpleCNN().to(device)
  model

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.SGD(model.parameters(), lr=lr)
  for epoch in range(number_of_epochs):  # Train for 5 epochs
    running_loss = 0.0
    for images, labels in train_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{number_of_epochs}], Loss: {running_loss / len(train_loader):.4f}')

  correct = 0
  total = 0
  with torch.no_grad():  # Disable gradient calculation for evaluation
    for images, labels in test_loader:
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

  print(f'Accuracy: {100 * correct / total:.2f}%')

In [37]:
train_and_evaluate(25, 0.05)

Epoch [1/25], Loss: 0.8796
Epoch [2/25], Loss: 0.5218
Epoch [3/25], Loss: 0.4435
Epoch [4/25], Loss: 0.3955
Epoch [5/25], Loss: 0.3626
Epoch [6/25], Loss: 0.3398
Epoch [7/25], Loss: 0.3207
Epoch [8/25], Loss: 0.2999
Epoch [9/25], Loss: 0.2876
Epoch [10/25], Loss: 0.2767
Epoch [11/25], Loss: 0.2649
Epoch [12/25], Loss: 0.2529
Epoch [13/25], Loss: 0.2442
Epoch [14/25], Loss: 0.2334
Epoch [15/25], Loss: 0.2271
Epoch [16/25], Loss: 0.2175
Epoch [17/25], Loss: 0.2098
Epoch [18/25], Loss: 0.2024
Epoch [19/25], Loss: 0.1940
Epoch [20/25], Loss: 0.1877
Epoch [21/25], Loss: 0.1810
Epoch [22/25], Loss: 0.1746
Epoch [23/25], Loss: 0.1655
Epoch [24/25], Loss: 0.1605
Epoch [25/25], Loss: 0.1523
Accuracy: 90.26%
